In [1]:
import pandas as pd
matches = pd.read_csv("epl_2025_matches.csv")

In [2]:
matches["date"] = pd.to_datetime(matches["date"])

num_cols = ["xG","xGA","npxG","npxGA","npxGD","deep","deep_allowed",
            "scored","missed","xpts","wins","draws","loses","pts",
            "ppda.att","ppda.def","ppda_allowed.att","ppda_allowed.def"]
matches[num_cols] = matches[num_cols].apply(pd.to_numeric, errors="coerce")

matches = matches.sort_values(["team","date"]).reset_index(drop=True)
matches.dtypes

h_a                         object
xG                         float64
xGA                        float64
npxG                       float64
npxGA                      float64
deep                         int64
deep_allowed                 int64
scored                       int64
missed                       int64
xpts                       float64
result                      object
date                datetime64[ns]
wins                         int64
draws                        int64
loses                        int64
pts                          int64
npxGD                      float64
team_id                      int64
team                        object
ppda.att                     int64
ppda.def                     int64
ppda_allowed.att             int64
ppda_allowed.def             int64
dtype: object

In [3]:
print("Missing values:\n", matches.isna().sum()[matches.isna().sum() > 0])
print("Teams:", matches["team"].nunique())                    # expect 20
print(matches.groupby("team").size().value_counts())          # expect 38 for all
print("Duplicates:", matches.duplicated(["team","date"]).sum())  # expect 0

# pts must agree with result
expected_pts = matches["result"].map({"w":3,"d":1,"l":0})
print("pts mismatches:", (matches["pts"] != expected_pts).sum())  # expect 0

print(matches["xpts"].describe())   # should sit between 0 and 3

Missing values:
 Series([], dtype: int64)
Teams: 20
38    20
Name: count, dtype: int64
Duplicates: 0
pts mismatches: 0
count    760.000000
mean       1.389289
std        0.823525
min        0.008000
25%        0.680475
50%        1.347400
75%        2.077225
max        2.984600
Name: xpts, dtype: float64


In [4]:
season = (matches.groupby("team")
  .agg(played=("result","count"),
       pts=("pts","sum"), xpts=("xpts","sum"),
       gf=("scored","sum"), ga=("missed","sum"),
       xg=("xG","sum"), xga=("xGA","sum"))
  .reset_index())

season["pts_luck"]  = season["pts"] - season["xpts"]     # negative = unlucky
season["table_rank"] = season["pts"].rank(ascending=False, method="min")
season["xpts_rank"]  = season["xpts"].rank(ascending=False, method="min")
season["rank_gap"]   = season["table_rank"] - season["xpts_rank"]   # positive = deserved a higher spot

season.sort_values("pts_luck")

,team,played,pts,xpts,gf,ga,xg,xga,pts_luck,table_rank,xpts_rank,rank_gap
19,Wolverhampton Wanderers,38,20,35.4384,27,68,38.441391,68.303994,-15.4384,20.0,19.0,1.0
10,Leeds,38,47,56.4957,49,56,59.770650,56.325910,-9.4957,14.0,8.0,6.0
7,Crystal Palace,38,45,53.8765,41,51,62.456488,60.046223,-8.8765,15.0,11.0,4.0
17,Tottenham,38,41,49.2538,48,57,46.796252,55.590944,-8.2538,17.0,13.0,4.0
6,Chelsea,38,52,58.8512,58,52,72.212460,58.117463,-6.8512,10.0,6.0,4.0
14,Newcastle United,38,49,54.8333,53,55,60.613249,57.312037,-5.8333,12.0,10.0,2.0
18,West Ham,38,39,43.4899,46,65,49.640421,68.116145,-4.4899,18.0,16.0,2.0
3,Brentford,38,53,57.2441,55,52,66.256674,59.110111,-4.2441,8.0,7.0,1.0
2,Bournemouth,38,57,60.3276,58,54,66.834552,56.769789,-3.3276,6.0,5.0,1.0
4,Brighton,38,53,55.3489,52,46,58.419679,53.751226,-2.3489,8.0,9.0,-1.0


In [5]:
import pandas as pd, numpy as np

matches["date"] = pd.to_datetime(matches["date"])

num_cols = ["xG","xGA","npxG","npxGA","npxGD","deep","deep_allowed",
            "scored","missed","xpts","wins","draws","loses","pts",
            "ppda.att","ppda.def","ppda_allowed.att","ppda_allowed.def"]
for c in num_cols:
    matches[c] = pd.to_numeric(matches[c], errors="coerce")

# Power BI dislikes dots in column names, so rename them
matches = matches.rename(columns={
    "ppda.att": "ppda_att", "ppda.def": "ppda_def",
    "ppda_allowed.att": "ppda_allowed_att", "ppda_allowed.def": "ppda_allowed_def"})

matches = matches.sort_values(["team", "date"]).reset_index(drop=True)

In [6]:
print("Teams:", matches.team.nunique())                          # expect 20
print(matches.groupby("team").size().value_counts())             # expect 38 for all
print("Missing values:", matches.isna().sum().sum())              # expect 0
print("pts mismatches:", (matches.pts != matches.result.map({"w":3,"d":1,"l":0})).sum())  # expect 0

Teams: 20
38    20
Name: count, dtype: int64
Missing values: 0
pts mismatches: 0


In [7]:
m = matches.copy()
m["match_no"]  = m.groupby("team").cumcount() + 1
m["cum_pts"]   = m.groupby("team")["pts"].cumsum()
m["cum_xpts"]  = m.groupby("team")["xpts"].cumsum()
m["cum_luck"]  = m["cum_pts"] - m["cum_xpts"]
m["match_luck"] = m["pts"] - m["xpts"]
m["venue"]     = m["h_a"].map({"h": "Home", "a": "Away"})
m["result_label"] = m["result"].map({"w": "Win", "d": "Draw", "l": "Loss"})
m["xg_roll5"]  = m.groupby("team")["xG"].transform(lambda s: s.rolling(5).mean())
m["xga_roll5"] = m.groupby("team")["xGA"].transform(lambda s: s.rolling(5).mean())

keep = ["team","team_id","match_no","date","venue","result_label","pts","xpts","match_luck",
        "cum_pts","cum_xpts","cum_luck","scored","missed","xG","xGA","npxG","npxGA","npxGD",
        "deep","deep_allowed","ppda_att","ppda_def","ppda_allowed_att","ppda_allowed_def",
        "xg_roll5","xga_roll5"]
m = m[keep].round(3)
m["date"] = m["date"].dt.date          # date only, so Power BI types it as Date
m.to_csv("matches.csv", index=False)
m.head()

,team,team_id,match_no,date,venue,result_label,pts,xpts,match_luck,cum_pts,...,npxGA,npxGD,deep,deep_allowed,ppda_att,ppda_def,ppda_allowed_att,ppda_allowed_def,xg_roll5,xga_roll5
0,Arsenal,83,1,2025-08-17,Away,Win,3,1.363,1.637,3,...,1.377,-0.046,3,7,259,23,166,21,NaN,NaN
1,Arsenal,83,2,2025-08-23,Home,Win,3,2.910,0.090,6,...,0.157,1.820,5,1,175,22,418,15,NaN,NaN
2,Arsenal,83,3,2025-08-31,Away,Loss,0,1.271,-1.271,6,...,0.481,0.004,5,4,296,23,226,15,NaN,NaN
3,Arsenal,83,4,2025-09-13,Home,Win,3,2.874,0.126,9,...,0.294,2.588,10,1,219,20,291,20,NaN,NaN
4,Arsenal,83,5,2025-09-21,Home,Draw,1,0.930,0.070,10,...,1.123,-0.404,10,3,150,22,323,19,1.631,0.686


In [8]:
season = (matches.groupby("team").agg(
    played=("result","count"), pts=("pts","sum"), xpts=("xpts","sum"),
    gf=("scored","sum"), ga=("missed","sum"), xg=("xG","sum"), xga=("xGA","sum"),
    npxg=("npxG","sum"), deep=("deep","sum"), deep_allowed=("deep_allowed","sum"),
    ppda_att=("ppda_att","sum"), ppda_def=("ppda_def","sum"),
).reset_index())

season["gd"]        = season.gf - season.ga
season["xgd"]       = season.xg - season.xga
season["ppda"]      = season.ppda_att / season.ppda_def     # lower = more pressing
season["pts_luck"]  = season.pts - season.xpts              # negative = unlucky
season["finishing"] = season.gf - season.xg                 # + = scoring more than chances deserve
season["defending"] = season.xga - season.ga                # + = conceding fewer than chances deserve

season = season.sort_values(["pts","gd","gf"], ascending=False).reset_index(drop=True)
season["table_rank"] = season.index + 1
season["xpts_rank"]  = season.xpts.rank(ascending=False, method="first").astype(int)
season["rank_gap"]   = season.table_rank - season.xpts_rank     # + = deserved a higher spot

season["luck_label"] = np.select(
    [season.pts_luck <= -3, season.pts_luck >= 3],
    ["Unlucky", "Lucky"], default="As expected")

# per-game versions (handy for scatter plots)
for c in ["pts","xpts","gf","ga","xg","xga"]:
    season[f"{c}_pg"] = season[c] / season.played

season = season.drop(columns=["ppda_att","ppda_def"]).round(2)
season.to_csv("season_table.csv", index=False)
season

,team,played,pts,xpts,gf,ga,xg,xga,npxg,deep,...,table_rank,xpts_rank,rank_gap,luck_label,pts_pg,xpts_pg,gf_pg,ga_pg,xg_pg,xga_pg
0,Arsenal,38,85,79.87,71,27,77.49,33.13,74.44,349,...,1,1,0,Lucky,2.24,2.10,1.87,0.71,2.04,0.87
1,Manchester City,38,78,73.95,77,35,79.36,46.49,76.31,409,...,2,2,0,Lucky,2.05,1.95,2.03,0.92,2.09,1.22
2,Manchester United,38,71,64.45,69,50,72.19,50.28,67.62,285,...,3,3,0,Lucky,1.87,1.70,1.82,1.32,1.90,1.32
3,Aston Villa,38,65,51.07,56,49,56.20,56.68,56.20,293,...,4,12,-8,Lucky,1.71,1.34,1.47,1.29,1.48,1.49
4,Liverpool,38,60,61.54,63,53,67.26,53.87,65.74,373,...,5,4,1,As expected,1.58,1.62,1.66,1.39,1.77,1.42
5,Bournemouth,38,57,60.33,58,54,66.83,56.77,62.27,275,...,6,5,1,Unlucky,1.50,1.59,1.53,1.42,1.76,1.49
6,Sunderland,38,54,42.03,42,48,43.14,60.34,39.33,212,...,7,18,-11,Lucky,1.42,1.11,1.11,1.26,1.14,1.59
7,Brighton,38,53,55.35,52,46,58.42,53.75,54.61,270,...,8,9,-1,As expected,1.39,1.46,1.37,1.21,1.54,1.41
8,Brentford,38,53,57.24,55,52,66.26,59.11,58.64,240,...,9,7,2,Unlucky,1.39,1.51,1.45,1.37,1.74,1.56
9,Chelsea,38,52,58.85,58,52,72.21,58.12,66.12,273,...,10,6,4,Unlucky,1.37,1.55,1.53,1.37,1.90,1.53


In [9]:
ha = matches.groupby(["team","h_a"]).agg(
    played=("result","count"), pts=("pts","sum"), xpts=("xpts","sum"),
    gf=("scored","sum"), ga=("missed","sum"), xg=("xG","sum"), xga=("xGA","sum")
).reset_index()
ha["venue"] = ha["h_a"].map({"h":"Home","a":"Away"})
ha["luck"]  = ha.pts - ha.xpts
ha = ha.drop(columns="h_a").round(2)
ha.to_csv("home_away.csv", index=False)

In [10]:
import os
for f in ["matches.csv","season_table.csv","home_away.csv"]:
    print(f, "→", os.path.getsize(f)//1024, "KB")
print("Saved in:", os.getcwd())

matches.csv → 98 KB
season_table.csv → 2 KB
home_away.csv → 2 KB
Saved in: c:\Users\linam\OneDrive\Desktop\projects\football project
